# Behavioral Funnel Analysis

## Por que usuários engajados não convertem?

Esta análise investiga o comportamento de usuários em um e-commerce, com foco em entender por que muitos usuários visualizam produtos, retornam à plataforma, mas não concluem compras.

A análise segue cinco etapas:

1. Problema
2. Exploração inicial
3. Descobertas comportamentais
4. Explicação do comportamento
5. Recomendações de negócio

In [1]:
import pandas as pd
import plotly.express as px


df = pd.read_csv(
    "../data/raw/events.csv",
    usecols=[
        "event_time",
        "event_type",
        "price",
        "user_id",
        "user_session",
        "product_id",
        "category_code",
        "brand"
    ],
    parse_dates=["event_time"]
)

df["event_type"] = df["event_type"].astype("category")
df["user_id"] = df["user_id"].astype("int32")
df["product_id"] = df["product_id"].astype("int32")
df["price"] = df["price"].astype("float32")

df = df[df["event_type"].isin(["view", "cart", "purchase"])]

df_sample = df.sample(frac=0.20, random_state=42)

## 1. Problema

A taxa de compra é baixa em relação ao volume de navegação.

A pergunta principal é:

> Os usuários não compram por falta de interesse ou existe alguma barreira entre a intenção e a decisão de compra?

In [ ]:
user_behavior = df_sample.groupby("user_id").agg(
    total_events=("event_type", "count"),
    total_views=("event_type", lambda x: (x == "view").sum()),
    total_carts=("event_type", lambda x: (x == "cart").sum()),
    total_purchases=("event_type", lambda x: (x == "purchase").sum()),
    total_sessions=("user_session", "nunique"),
    avg_price=("price", "mean")
).reset_index()

user_behavior["is_buyer"] = user_behavior["total_purchases"] > 0

buyer_rate = user_behavior["is_buyer"].value_counts(normalize=True) * 100
buyer_rate

## 2. Exploração inicial

A maior parte dos usuários não realiza compra.  
Isso indica um funil com forte volume de navegação, mas baixa conversão.

In [ ]:
event_counts = (
    df_sample["event_type"]
    .value_counts()
    .reset_index()
)

event_counts.columns = ["event_type", "count"]

fig = px.bar(
    event_counts,
    x="event_type",
    y="count",
    title="A maioria dos eventos é de visualização"
)

fig.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis_title="Tipo de evento",
    yaxis_title="Quantidade"
)

fig.show()


## 3. Tempo até compra

Agora vamos investigar quanto tempo os usuários levam entre a primeira visualização e a primeira compra.

In [ ]:
df_sorted = df_sample.sort_values(["user_id", "event_time"])

first_view = (
    df_sorted[df_sorted["event_type"] == "view"]
    .groupby("user_id")["event_time"]
    .min()
)

first_purchase = (
    df_sorted[df_sorted["event_type"] == "purchase"]
    .groupby("user_id")["event_time"]
    .min()
)

time_to_purchase = (first_purchase - first_view).dropna()

time_df = time_to_purchase.dt.total_seconds().div(60).reset_index()
time_df.columns = ["user_id", "time_to_purchase_minutes"]

time_df_filtered = time_df[
    (time_df["time_to_purchase_minutes"] >= 0) &
    (time_df["time_to_purchase_minutes"] <= 1440)
]

time_df_filtered["time_to_purchase_minutes"].describe()

count    36867.000000
mean       176.954404
std        362.903925
min          0.133333
25%          3.483333
50%         10.366667
75%         84.250000
max       1440.000000
Name: time_to_purchase_minutes, dtype: float64

In [ ]:
fig = px.histogram(
    time_df_filtered,
    x="time_to_purchase_minutes",
    nbins=50,
    title="Metade das compras acontece rapidamente"
)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis_title="Tempo até compra (minutos)",
    yaxis_title="Quantidade de usuários"
)

fig.show()

### Insight

A mediana do tempo até compra indica que muitos usuários compram rapidamente após a primeira visualização.

Por outro lado, a média tende a ser maior por causa de uma cauda longa: alguns usuários demoram muitas horas até comprar.

Isso sugere dois comportamentos:

- compradores de decisão rápida
- compradores exploratórios, que pesquisam mais antes de converter

In [ ]:
median_time = time_df_filtered["time_to_purchase_minutes"].median()

fast_buyers = time_df_filtered[
    time_df_filtered["time_to_purchase_minutes"] <= median_time
]

slow_buyers = time_df_filtered[
    time_df_filtered["time_to_purchase_minutes"] > median_time
]

print("Mediana em minutos:", median_time)
print("Compradores rápidos:", len(fast_buyers))
print("Compradores lentos:", len(slow_buyers))
print("Percentual rápidos:", len(fast_buyers) / len(time_df_filtered) * 100)
print("Percentual lentos:", len(slow_buyers) / len(time_df_filtered) * 100)

Mediana em minutos: 10.366666666666667
Compradores rápidos: 18436
Compradores lentos: 18431
Percentual rápidos: 50.00678113217782
Percentual lentos: 49.99321886782217


In [ ]:
fast_ids = fast_buyers["user_id"]
slow_ids = slow_buyers["user_id"]

fast_behavior = user_behavior[user_behavior["user_id"].isin(fast_ids)]
slow_behavior = user_behavior[user_behavior["user_id"].isin(slow_ids)]

comparison_df = pd.DataFrame({
    "Compradores rápidos": fast_behavior[
        ["total_views", "total_carts", "total_sessions", "total_events"]
    ].mean(),
    "Compradores lentos": slow_behavior[
        ["total_views", "total_carts", "total_sessions", "total_events"]
    ].mean()
})

comparison_df

,Compradores rápidos,Compradores lentos
total_views,4.456227,9.063534
total_carts,0.458451,0.614671
total_sessions,2.812649,4.767131
total_events,6.260414,11.061961


## 4. Compradores rápidos vs compradores exploratórios

Compradores lentos tendem a apresentar mais visualizações, mais eventos e mais sessões antes da compra.

Isso indica uma jornada mais exploratória e comparativa.

In [ ]:
comparison_plot = comparison_df.reset_index()
comparison_plot.columns = ["métrica", "rápidos", "lentos"]

comparison_melted = comparison_plot.melt(
    id_vars="métrica",
    var_name="grupo",
    value_name="valor"
)

fig = px.bar(
    comparison_melted,
    x="métrica",
    y="valor",
    color="grupo",
    barmode="group",
    title="Compradores lentos exploram mais antes de comprar"
)

fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis_title="",
    yaxis_title="Média por usuário"
)

fig.show()

fig.write_image("../output/charts/time_to_purchase.png")

## 5. Segmentação comportamental

A partir do comportamento de navegação e compra, os usuários foram classificados em quatro personas:

- Usuário casual
- Interessado sem conversão
- Comprador impulsivo
- Comprador explorador

In [ ]:
def classify_user(row):
    if row["total_purchases"] == 0 and row["total_views"] >= 5:
        return "Interessado sem conversão"

    elif row["total_purchases"] > 0 and row["total_sessions"] <= 2:
        return "Comprador impulsivo"

    elif row["total_purchases"] > 0 and row["total_sessions"] > 2:
        return "Comprador explorador"

    else:
        return "Usuário casual"


user_behavior["persona"] = user_behavior.apply(classify_user, axis=1)

persona_analysis = user_behavior.groupby("persona").agg({
    "total_views": "mean",
    "total_sessions": "mean",
    "avg_price": "mean",
    "total_purchases": "mean"
}).round(2)

persona_analysis

,total_views,total_sessions,avg_price,total_purchases
persona,,,,
Comprador explorador,14.03,7.61,299.769989,1.56
Comprador impulsivo,1.73,1.44,311.570007,1.06
Interessado sem conversão,11.43,4.83,285.029999,0.00
Usuário casual,1.77,1.34,320.049988,0.00


In [ ]:
persona_counts = (
    user_behavior["persona"]
    .value_counts()
    .reset_index()
)

persona_counts.columns = ["persona", "count"]

persona_focus = persona_counts[
    persona_counts["persona"] != "Usuário casual"
]

fig = px.bar(
    persona_focus,
    x="persona",
    y="count",
    title="Usuários altamente engajados não convertem"
)

fig.update_traces(marker_color="lightgray")

fig.data[0].marker.color = [
    "#d62728" if x == "Interessado sem conversão" else "lightgray"
    for x in persona_focus["persona"]
]

fig.update_layout(
    showlegend=False,
    xaxis_title="",
    yaxis_title="",
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_yaxes(showgrid=False)

fig.add_annotation(
    x="Interessado sem conversão",
    y=persona_focus.loc[
        persona_focus["persona"] == "Interessado sem conversão",
        "count"
    ].iloc[0],
    text="Usuários engajados sem compra",
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-50
)

fig.show()

fig.write_image("../output/charts/personas.png")

### Insight

Os usuários mais engajados não são necessariamente os que mais compram.

O grupo “Interessado sem conversão” apresenta comportamento de exploração intensa, com várias visualizações e sessões, mas sem conclusão de compra.

Isso sugere que o problema pode não estar apenas na atração de usuários, mas na transformação de interesse em decisão.

## 6. Categorias mais exploradas por usuários sem conversão

Vamos investigar quais categorias concentram o comportamento dos usuários interessados que não compraram.

In [ ]:
interested_no_purchase = user_behavior[
    user_behavior["persona"] == "Interessado sem conversão"
]

non_buyers_ids = interested_no_purchase["user_id"]

non_buyers_events = df_sample[
    df_sample["user_id"].isin(non_buyers_ids)
]

top_categories = (
    non_buyers_events["category_code"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_categories.columns = ["category_code", "count"]

top_categories

,category_code,count
0,electronics.smartphone,1154216
1,electronics.clocks,156346
2,computers.notebook,146589
3,electronics.video.tv,125453
4,electronics.audio.headphone,108817
5,appliances.kitchen.refrigerators,108285
6,appliances.kitchen.washer,100387
7,appliances.environment.vacuum,97785
8,apparel.shoes,97392
9,auto.accessories.player,57178


In [ ]:
fig = px.bar(
    top_categories,
    x="count",
    y="category_code",
    orientation="h",
    title="Usuários sem conversão exploram categorias de decisão complexa"
)

fig.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis_title="Visualizações",
    yaxis_title="",
    yaxis=dict(autorange="reversed")
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

fig.write_image("../output/charts/low_conversion_products.png")

### Insight

As categorias mais exploradas pelos usuários sem conversão concentram produtos como smartphones, notebooks e eletrônicos.

Esses produtos normalmente envolvem comparação, pesquisa de preço, avaliação técnica e múltiplas visitas antes da decisão.

Isso reforça a hipótese de uma jornada mais exploratória e menos impulsiva.

## 7. Produtos com alto interesse e baixa conversão

Agora buscamos produtos que recebem muitas visualizações, mas apresentam baixa taxa de compra.

In [ ]:
product_views = (
    df_sample[df_sample["event_type"] == "view"]
    ["product_id"]
    .value_counts()
)

product_purchases = (
    df_sample[df_sample["event_type"] == "purchase"]
    ["product_id"]
    .value_counts()
)

product_conversion = pd.DataFrame({
    "views": product_views,
    "purchases": product_purchases
}).fillna(0)

product_conversion["conversion_rate"] = (
    product_conversion["purchases"] / product_conversion["views"]
) * 100

product_conversion = product_conversion.reset_index()
product_conversion.columns = ["product_id", "views", "purchases", "conversion_rate"]

high_interest_low_conversion = (
    product_conversion[
        (product_conversion["views"] >= product_conversion["views"].quantile(0.90)) &
        (product_conversion["conversion_rate"] <= product_conversion["conversion_rate"].quantile(0.25))
    ]
    .sort_values("views", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

high_interest_low_conversion

,product_id,views,purchases,conversion_rate
0,10300835,2842.0,0.0,0.0
1,15400069,2525.0,0.0,0.0
2,10700979,1739.0,0.0,0.0
3,1003571,1225.0,0.0,0.0
4,15100376,1179.0,0.0,0.0
5,1480492,1157.0,0.0,0.0
6,1003475,1055.0,0.0,0.0
7,15700000,1016.0,0.0,0.0
8,52100005,1004.0,0.0,0.0
9,5100797,911.0,0.0,0.0


In [ ]:
fig = px.bar(
    high_interest_low_conversion,
    x="views",
    y=high_interest_low_conversion["product_id"].astype(str),
    orientation="h",
    title="Produtos atraem usuários, mas não convertem"
)

fig.update_traces(marker_color="lightgray")

colors = ["#d62728"] + ["lightgray"] * (len(high_interest_low_conversion) - 1)
fig.update_traces(marker_color=colors)

fig.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis_title="Visualizações",
    yaxis_title="",
    yaxis=dict(autorange="reversed")
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()
fig.write_image("../output/charts/low_conversion_products.png")

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


### Insight

Alguns produtos concentram muitas visualizações, mas apresentam baixa taxa de conversão.

Esse comportamento sugere que o usuário demonstra interesse, mas encontra alguma barreira antes da decisão de compra.

A oportunidade está em investigar esses produtos individualmente: preço, descrição, avaliações, confiança, comparação com concorrentes e clareza da oferta.

## 8. Recomendações

Os dados sugerem que o principal desafio pode não estar na atração de usuários, mas na transformação de interesse em decisão de compra.

Os usuários mais engajados concentraram sua navegação principalmente em categorias de maior complexidade de decisão, como smartphones, notebooks e eletrônicos, indicando uma jornada mais exploratória e comparativa.

Com base nisso, possíveis ações incluem:

- destacar avaliações e prova social
- melhorar mecanismos de comparação
- reduzir fricção no funil
- criar incentivos de conversão
- simplificar a descoberta de produtos
- recomendar produtos com maior confiança de compra